# SearchLite-RL FINAL: freeze -> sealed hard evaluation -> paired RL analysis -> submission package
Branch `experiment/headroom-grpo`. Run cells in order. Every code cell is `%%bash` with `set -euo pipefail`. Nothing is deleted or overwritten; outputs live on Drive (`MyDrive/SearchLite-RL`).

| Cell | What | GPU | Time (L4) |
|---|---|---|---|
| 4 | inventory of artifacts + claims check against the saved JSONs | no | ~2 min |
| 5 | format control (M2 under `typed` vs `typed_abstain`, dev splits only) | yes | ~6 min |
| 6 | **dry-run** of the freeze: shows the decision, writes nothing | no | <1 min |
| 7 | **FREEZE** (immutable, read-only). Run once, only after reading cell 6 | no | <1 min |
| 8 | sealed `hard_final` evaluation (base, M1, M2, M2-typed, M3/M4/M4g x seeds), once | yes | ~25 min |
| 9 | paired RL-improvement analysis, failure modes, figures, case studies, report, checklist | no | ~3 min |
| 10 | copy the package to Drive (no overwrite) | no | <1 min |

If a gate or a check exits non-zero, paste the output and do not continue past it. No training happens in this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -euo pipefail
cd /content
if [ -d Pilot-project ]; then cd Pilot-project && git fetch -q && git checkout -q experiment/headroom-grpo && git pull -q; else git clone -q -b experiment/headroom-grpo https://github.com/aasritha-oql7872/Pilot-project.git; fi
cd /content/Pilot-project
python -c "import trl,peft,transformers" 2>/dev/null && echo "dependencies already installed" || { pip install -q -r requirements.txt; pip uninstall -y -q torchvision torchaudio torchao; echo "INSTALLED: restart the session (Runtime > Restart session), then continue with the next cell"; }
git log --oneline -3

In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
D=/content/drive/MyDrive/SearchLite-RL
mkdir -p outputs $D/study_sft $D/study_hard
for s in study_sft study_hard; do ln -sfn $D/$s outputs/$s; done
mkdir -p outputs/main_v2
[ -f outputs/main_v2/adapter/adapter_config.json ] || cp -rn $D/outputs/main_v2/adapter outputs/main_v2/ 2>/dev/null || cp -rn $D/main_v2/adapter outputs/main_v2/ 2>/dev/null || echo "condition A adapter not on Drive (M1 would be retrained by run_sft_study.sh)"
nvidia-smi -L
ls outputs/study_sft | head -40
python -m pytest -q 2>&1 | tail -3

In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
S=outputs/study_hard
echo "--- artifact inventory (Drive) ---"
for f in outputs/study_sft/B/adapter/adapter_config.json outputs/main_v2/adapter/adapter_config.json \
         $S/eval_hard_dev_M2.json $S/eval_twohop_dev_M2.json $S/eval_dev_M2.json $S/decision_headroom.json $S/decision_signal.json \
         $S/summary_hard_dev.md $S/lineage.json $S/traj_stats.json; do [ -e "$f" ] && echo "OK       $f" || echo "MISSING  $f"; done
for v in M3 M4 M4g; do for s in 42 43; do
  [ -f $S/${v}_s$s/adapter/adapter_config.json ] && echo "OK       ${v}_s$s adapter" || echo "MISSING  ${v}_s$s adapter"; done; done
ls $S | grep -c crashed || true
echo "--- hard_final evaluated before? (must be empty) ---"
ls $S/eval_hard_final_* $S/final_*.log 2>/dev/null || echo "none: hard_final is untouched"
echo "--- claims from the earlier summary vs saved JSONs ---"
MODE=pre bash run_final_analysis.sh
cat submission/tables/claims_check.md


In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
# inference only, dev splits: M2 under the reminder it was trained with ("typed") vs the hard benchmark's "typed_abstain"
bash run_format_control.sh


In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
# DRY RUN: prints the decision and the evidence behind it; writes nothing. Read it before cell 7.
python -m src.freeze_hard --dry-run


In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
# FREEZE: writes outputs/study_hard/FROZEN_HARD.json (read-only). Refuses to run twice. Do not run this until you have read cell 6.
python -m src.freeze_hard


In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
# SEALED hard_final evaluation. Resumable: finished evaluations are skipped, nothing is overwritten.
bash run_hard_final.sh 2>&1 | tee -a outputs/study_hard/final_$(date +%Y%m%d_%H%M).log | tail -60
cat outputs/study_hard/summary_hard_final.md


In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
MODE=post bash run_final_analysis.sh 2>&1 | tail -60


In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
D=/content/drive/MyDrive/SearchLite-RL
mkdir -p $D/submission_package
cp -rn submission/. $D/submission_package/ 
cp -n ../searchlite_rl_submission.zip $D/ 2>/dev/null || cp -n searchlite_rl_submission.zip $D/ 2>/dev/null || true
echo "copied (existing files kept) -> $D/submission_package"
cat submission/STATUS_CHECKLIST.md
